# no-relu-on-final-layer — worked example 1: strip a stray ReLU from a regression head

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `no-relu-on-final-layer`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The final layer of a regressor (or classifier producing logits) must be bare: any non-negative-clipping activation like ReLU after it prevents the model from ever producing negative outputs. The fix reuses the existing weights but drops the trailing activation.

## Worked solution

We are given `BrokenRegressor`, whose forward is `fc1 -> ReLU -> fc2 -> ReLU`, where the second ReLU is the bug: a regression target can be negative, but the clipped output never is. We build `FixedRegressor` that holds references to the same `fc1` and `fc2` modules (no re-initialization, so the learned weights survive) and applies ReLU only between them, returning the raw output of `fc2`. We seed, build the broken model, fix it, run a random batch through the fixed model, and print the fraction of negative outputs, which is now well above zero, proving the final ReLU is gone.

In [ ]:
import torch as t
import torch.nn as nn
import torch.nn.functional as F

t.manual_seed(0)

class BrokenRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(6, 12)
        self.fc2 = nn.Linear(12, 1)
    def forward(self, x):
        return F.relu(self.fc2(F.relu(self.fc1(x))))  # stray final ReLU

def fix_regressor(broken):
    class FixedRegressor(nn.Module):
        def __init__(self, fc1, fc2):
            super().__init__()
            self.fc1 = fc1
            self.fc2 = fc2
        def forward(self, x):
            return self.fc2(F.relu(self.fc1(x)))  # no final ReLU
    return FixedRegressor(broken.fc1, broken.fc2)

broken = BrokenRegressor()
fixed = fix_regressor(broken)
x = t.randn(64, 6)
frac_neg = (fixed(x) < 0).float().mean().item()
print('fraction negative (fixed):', round(frac_neg, 3))